# FaithLM cho tiếng Việt — Đánh giá độ trung thực của giải thích LLM

Notebook này tái lập và mở rộng **FaithLM** (EACL 2026) trên **XCOPA tiếng Việt**.

| | |
|---|---|
| **Predictor** (mô hình cần giải thích) | Qwen3-4B-Instruct |
| **Explainer** (mô hình sinh giải thích) | DeepSeek V4 Pro (API) |
| **Dữ liệu** | XCOPA-vi (500 câu, suy luận nhân quả) + COPA-en để đối chiếu |
| **Chỉ số** | Faithfulness = \|acc(giải thích thật) − acc(giải thích đối nghịch)\| |

### Đóng góp của chúng tôi so với mã nguồn gốc

1. **Chấm điểm bằng log-probability.** Bản gốc so khớp chuỗi trên **một** câu hỏi, nên điểm chỉ nhận giá trị 0.0 hoặc 1.0 và hiệu số gần như luôn bằng 0 — LLM-OPT không có tín hiệu để tối ưu. Chúng tôi tính log-prob mà predictor gán cho từng lựa chọn, cho điểm **liên tục** trong [0,1] chỉ với một forward pass.
2. **Chỉ số `symmetric`.** Bản gốc so sánh *không có gợi ý* với *gợi ý đối nghịch*, nên giải thích thật không bao giờ được đưa vào predictor. Chúng tôi bổ sung biến thể đưa gợi ý vào **cả hai** nhánh, tách bạch tác động của việc *đảo nghĩa* khỏi tác động của việc *có gợi ý*.
3. **Baseline không dùng LLM** (phủ định theo luật, đồng nhất, ngẫu nhiên) để trả lời: bước sinh đối nghịch có thực sự cần LLM không?
4. **Chạy lại được sau khi mất session** — quan trọng với giới hạn 12 giờ của Kaggle.

> **Trước khi chạy**: bật **GPU** (Settings → Accelerator) và **Internet** (Settings → Internet on).
> Trên Kaggle, thêm `DEEPSEEK_API_KEY` qua *Add-ons → Secrets*.

## 1. Cài đặt thư viện và cấu hình môi trường

In [ ]:
%pip install -q "transformers>=4.44" "datasets>=2.19" accelerate bitsandbytes \
    "openai>=1.30" scikit-learn pyyaml matplotlib pandas

In [ ]:
# Lấy mã nguồn. Notebook không phụ thuộc bất kỳ đường dẫn cục bộ nào.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/YOUR_TEAM/NLP_Final.git"  # <-- đổi thành repo của nhóm
REPO_DIR = "NLP_Final"

if pathlib.Path("faithlm").is_dir():
    PROJECT_ROOT = os.getcwd()                    # đang chạy ngay trong repo
elif pathlib.Path(REPO_DIR).is_dir():
    PROJECT_ROOT = os.path.abspath(REPO_DIR)      # đã clone từ lần chạy trước
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    PROJECT_ROOT = os.path.abspath(REPO_DIR)

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Thư mục dự án:", PROJECT_ROOT)

In [ ]:
# Khóa API: đọc từ Kaggle Secrets, Colab userdata, hoặc biến môi trường.
import os

def load_api_key():
    if os.environ.get("DEEPSEEK_API_KEY"):
        return "biến môi trường"
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["DEEPSEEK_API_KEY"] = UserSecretsClient().get_secret("DEEPSEEK_API_KEY")
        return "Kaggle Secrets"
    except Exception:
        pass
    try:
        from google.colab import userdata
        os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
        return "Colab userdata"
    except Exception:
        pass
    return None

source = load_api_key()
print(f"Khóa DeepSeek: {'đã nạp từ ' + source if source else 'CHƯA CÓ — chỉ chạy được baseline không dùng LLM'}")

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có (sẽ rất chậm)'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Tải bộ dữ liệu

In [ ]:
from faithlm import datasets as fl_datasets

vi = fl_datasets.load("xcopa_vi", lang="vi", split="test")
en = fl_datasets.load("copa_en", split="train")
print(f"XCOPA-vi : {len(vi)} câu")
print(f"COPA-en  : {len(en)} câu")

## 3. Khám phá và tiền xử lý dữ liệu

In [ ]:
print("=== MỘT MẪU XCOPA TIẾNG VIỆT ===\n")
print(vi[0].question)
print(f"\nĐáp án đúng: {vi[0].answer}")
print(f"Các lựa chọn: {vi[0].choices}")

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.DataFrame([{
    "tách_từ": len(ex.question.split()),
    "ký_tự": len(ex.question),
    "độ_dài_đáp_án": len(ex.answer.split()),
    "loại": "cause" if "cause" in ex.question else "effect",
} for ex in vi])

print(df.describe().round(1).to_string())
print(f"\nPhân bố loại câu hỏi:\n{df['loại'].value_counts().to_string()}")

# Kiểm tra cân bằng nhãn — COPA cân bằng theo thiết kế; lệch là dấu hiệu lỗi tải dữ liệu.
first_choice = sum(1 for ex in vi if ex.answer == ex.choices[0])
print(f"\nNhãn chọn lựa chọn 1: {first_choice}/{len(vi)} ({first_choice/len(vi):.1%})")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df["tách_từ"], bins=25, color="#4C78A8", edgecolor="white")
axes[0].set_title("Độ dài câu hỏi (số từ)"); axes[0].set_xlabel("số từ")
axes[1].hist(df["độ_dài_đáp_án"], bins=15, color="#F58518", edgecolor="white")
axes[1].set_title("Độ dài đáp án (số từ)"); axes[1].set_xlabel("số từ")
plt.tight_layout(); plt.show()

## 4. Nạp mô hình

Nạp predictor **một lần** rồi dùng lại cho mọi biến thể — nạp lại mỗi lần sẽ tốn phần lớn thời gian session.

In [ ]:
import torch
from faithlm import predictors, explainers, from_dict

# GPU dưới 12GB thì bật lượng tử hóa 4-bit; T4/P100 16GB dùng bf16 cho nhanh.
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
USE_4BIT = vram_gb < 12
print(f"VRAM {vram_gb:.1f} GB → {'4-bit NF4' if USE_4BIT else 'bf16'}")

cfg = from_dict({
    "dataset":   {"name": "xcopa_vi", "lang": "vi", "split": "test"},
    "predictor": {"name": "qwen", "model_id": "Qwen/Qwen3-4B-Instruct-2507",
                  "load_in_4bit": USE_4BIT},
    "explainer": {"name": "deepseek"},
    "metric":    {"name": "paper", "scorer": "logprob"},
    "run":       {"pipeline": "local", "ques_idx_end": 30, "xai_iter": 5,
                  "output_dir": "./results", "resume": True},
})

predictor = predictors.build(cfg.predictor)
explainer = explainers.build(cfg.explainer)
print("Đã nạp xong.")

### Kiểm chứng: điểm số có thực sự liên tục không?

Đây là kiểm tra quan trọng nhất của cải tiến thứ nhất. Nếu chấm bằng so khớp chuỗi trên một câu, giá trị chỉ có thể là 0.0 hoặc 1.0.

In [ ]:
from faithlm.metrics import score_arm
from faithlm.prompts import TASK_INSTRUCTION_MC

print(f"{'câu':>4} | {'log-prob':>9} | {'so khớp chuỗi':>14}")
print("-" * 34)
for i in range(6):
    lp = score_arm(predictor, vi[i], TASK_INSTRUCTION_MC, None, "logprob").true_arm
    em = score_arm(predictor, vi[i], TASK_INSTRUCTION_MC, None, "exact_match").true_arm
    print(f"{i:>4} | {lp:>9.4f} | {em:>14.1f}")
print("\nCột log-prob nhận giá trị liên tục; cột so khớp chuỗi chỉ có 0 hoặc 1.")

## 5. Chạy thực nghiệm

Mỗi biến thể ghi kết quả xuống đĩa theo từng câu. Nếu session bị ngắt, chạy lại cell này sẽ **tiếp tục từ chỗ dừng**.

In [ ]:
from faithlm import run_experiment
import copy

N_QUESTIONS = 30   # tăng lên 200 cho lần chạy đầy đủ
N_ITER = 5

def variant(**overrides):
    c = copy.deepcopy(cfg)
    c.run.ques_idx_end, c.run.xai_iter = N_QUESTIONS, N_ITER
    for section, values in overrides.items():
        for k, v in values.items():
            setattr(getattr(c, section), k, v)
    return c

VARIANTS = {
    "FaithLM (gốc)":        variant(metric={"name": "paper"}),
    "FaithLM (symmetric)":  variant(metric={"name": "symmetric"}),
    "Baseline: phủ định":   variant(explainer={"name": "baseline_negation"}, run={"xai_iter": 1}),
    "Baseline: đồng nhất":  variant(explainer={"name": "baseline_identity"}, run={"xai_iter": 1}),
    "Baseline: ngẫu nhiên": variant(explainer={"name": "baseline_shuffle"}, run={"xai_iter": 1}),
}

summaries = {}
for name, vcfg in VARIANTS.items():
    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")
    # Baseline không gọi API nên dùng explainer riêng; các biến thể khác dùng chung.
    exp = explainer if vcfg.explainer.name == "deepseek" else explainers.build(vcfg.explainer)
    summaries[name] = run_experiment(vcfg, predictor=predictor, explainer=exp)

## 6. Đánh giá và so sánh với mô hình cơ sở

In [ ]:
import pandas as pd

table = pd.DataFrame([{
    "Phương pháp": name,
    "Độ chính xác tác vụ": round(s["task_accuracy"], 3),
    "Faithfulness TB": round(s["mean_best_faithfulness"], 4),
    "Số câu": s["questions"],
    "Lỗi": s["failed"],
} for name, s in summaries.items()])

display(table)

best = table.loc[table["Faithfulness TB"].idxmax(), "Phương pháp"]
print(f"\nĐiểm faithfulness cao nhất: {best}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#4C78A8" if "FaithLM" in n else "#BAB0AC" for n in table["Phương pháp"]]
ax.barh(table["Phương pháp"], table["Faithfulness TB"], color=colors, edgecolor="white")
ax.set_xlabel("Điểm faithfulness trung bình")
ax.set_title("FaithLM so với các baseline không dùng LLM (XCOPA tiếng Việt)")
ax.invert_yaxis()
for i, v in enumerate(table["Faithfulness TB"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

### Đối chiếu chéo ngôn ngữ: tiếng Việt so với tiếng Anh

Cùng mô hình, cùng chỉ số, chỉ khác ngôn ngữ. Chênh lệch cho biết pipeline giải thích suy giảm bao nhiêu khi rời khỏi tiếng Anh.

In [ ]:
cfg_en = variant(dataset={"name": "copa_en", "lang": "en", "split": "train"})
summaries["FaithLM (COPA tiếng Anh)"] = run_experiment(cfg_en, predictor=predictor, explainer=explainer)

vi_score = summaries["FaithLM (gốc)"]["mean_best_faithfulness"]
en_score = summaries["FaithLM (COPA tiếng Anh)"]["mean_best_faithfulness"]
vi_acc = summaries["FaithLM (gốc)"]["task_accuracy"]
en_acc = summaries["FaithLM (COPA tiếng Anh)"]["task_accuracy"]

print(f"{'':12s} {'độ chính xác':>13s} {'faithfulness':>14s}")
print(f"{'Tiếng Việt':12s} {vi_acc:>13.3f} {vi_score:>14.4f}")
print(f"{'Tiếng Anh':12s} {en_acc:>13.3f} {en_score:>14.4f}")
print(f"{'Chênh lệch':12s} {en_acc - vi_acc:>13.3f} {en_score - vi_score:>14.4f}")

## 7. Phân tích lỗi

In [ ]:
import json, glob, os

variant_dir = os.path.join("results", VARIANTS["FaithLM (gốc)"].variant_id(), "local")
records = []
for path in sorted(glob.glob(os.path.join(variant_dir, "sample-*.json"))):
    with open(path, encoding="utf-8") as f:
        records.append(json.load(f))
records = [r for r in records if not r.get("error")]
print(f"Đã nạp {len(records)} kết quả\n")

# Phân loại lỗi: dự đoán sai vs. giải thích không trung thực là hai lỗi khác nhau.
buckets = {
    "Đúng + giải thích trung thực":   [r for r in records if r["correct"] and r["best_score"] > 0.1],
    "Đúng + giải thích không trung thực": [r for r in records if r["correct"] and r["best_score"] <= 0.1],
    "Sai + giải thích trung thực":     [r for r in records if not r["correct"] and r["best_score"] > 0.1],
    "Sai + giải thích không trung thực":  [r for r in records if not r["correct"] and r["best_score"] <= 0.1],
}
for label, group in buckets.items():
    pct = len(group) / len(records) * 100 if records else 0
    print(f"{label:38s} {len(group):>3d}  ({pct:>5.1f}%)")

In [ ]:
# Trường hợp đáng chú ý nhất: predictor trả lời đúng nhưng giải thích không
# bám vào lý do thật — đây chính là hiện tượng FaithLM muốn phát hiện.
group = buckets["Đúng + giải thích không trung thực"]
for r in group[:2]:
    print("=" * 70)
    print(r["question"][:280])
    print(f"\nĐáp án đúng     : {r['gold_answer']}")
    print(f"Mô hình trả lời : {r['model_answer'][:120]}")
    print(f"Faithfulness    : {r['best_score']}")
    print(f"Giải thích      : {r['best_explanation'][:280]}")
    print()
if not group:
    print("Không có trường hợp nào thuộc nhóm này trong mẫu hiện tại.")

In [ ]:
# LLM-OPT có thực sự cải thiện qua các vòng lặp không?
import matplotlib.pyplot as plt

curves = [[it["score"] for it in r["iterations"]] for r in records if len(r["iterations"]) > 1]
if curves:
    max_len = max(len(c) for c in curves)
    means = [
        sum(c[i] for c in curves if len(c) > i) / sum(1 for c in curves if len(c) > i)
        for i in range(max_len)
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    for c in curves[:25]:
        ax.plot(range(len(c)), c, color="#BAB0AC", alpha=0.35, linewidth=1)
    ax.plot(range(max_len), means, color="#4C78A8", linewidth=2.5, marker="o", label="trung bình")
    ax.set_xlabel("Vòng lặp LLM-OPT"); ax.set_ylabel("Điểm faithfulness")
    ax.set_title("Quỹ đạo tối ưu hóa giải thích"); ax.legend()
    plt.tight_layout(); plt.show()
    print(f"Vòng đầu: {means[0]:.4f} → vòng cuối: {means[-1]:.4f}  (thay đổi {means[-1] - means[0]:+.4f})")
else:
    print("Chưa đủ vòng lặp để vẽ quỹ đạo — tăng xai_iter.")

## 8. Demo trên dữ liệu tiếng Việt mới

In [ ]:
# Câu tự viết, không nằm trong XCOPA — kiểm tra pipeline trên dữ liệu hoàn toàn mới.
from faithlm.datasets import Example, _render_copa_style

def make_example(premise, question_type, choice_a, choice_b, gold_index):
    choices = [choice_a, choice_b]
    return Example(
        question=_render_copa_style(premise, question_type, choices),
        answer=choices[gold_index],
        choices=choices,
    )

demo = [
    make_example("Trời mưa rất to suốt cả buổi sáng.", "effect",
                 "Sân trường ngập nước.", "Học sinh ra sân chơi bóng.", 0),
    make_example("Cô ấy quên mang ví khi đi siêu thị.", "effect",
                 "Cô ấy mua rất nhiều đồ.", "Cô ấy phải để lại hàng ở quầy.", 1),
    make_example("Anh ta bị cảnh sát phạt tiền.", "cause",
                 "Anh ta vượt đèn đỏ.", "Anh ta dừng xe đúng vạch.", 0),
]

for ex in demo:
    print(ex.question)
    print(f"→ đáp án đúng: {ex.answer}\n")

In [ ]:
from faithlm import pipelines

demo_cfg = variant(run={"ques_idx_end": len(demo), "xai_iter": 3,
                        "output_dir": "./results_demo", "resume": False})
demo_results = pipelines.run_local(demo_cfg, demo, predictor, explainer)

for ex, r in zip(demo, demo_results):
    print("=" * 70)
    print(f"Tiền đề     : {ex.question.split('### Premise: ')[1].split(chr(10))[0]}")
    print(f"Đáp án đúng : {r['gold_answer']}")
    print(f"Mô hình chọn: {r['model_answer'][:100]}   {'✓' if r['correct'] else '✗'}")
    print(f"Faithfulness: {r['best_score']}")
    print(f"Giải thích  : {r['best_explanation'][:300]}")
    print()

## 9. Kết luận

Điền các con số thực tế sau khi chạy xong:

| Câu hỏi | Kết quả |
|---|---|
| Log-prob có cho điểm liên tục không? | (xem mục 4) |
| `symmetric` khác `paper` bao nhiêu? | (xem mục 6) |
| FaithLM có vượt baseline phủ định không? | (xem mục 6) |
| Khoảng cách Việt–Anh? | (xem mục 6) |
| LLM-OPT có cải thiện qua vòng lặp? | (xem mục 7) |

**Hạn chế cần nêu trong báo cáo:**

- XCOPA-vi là bản dịch từ COPA tiếng Anh, nên có thể mang dấu vết dịch thuật (*translationese*) thay vì phản ánh cách diễn đạt nhân quả tự nhiên trong tiếng Việt.
- Chấm bằng log-prob chỉ áp dụng được cho predictor chạy cục bộ; predictor qua API phải dùng `scorer="exact_match"`.
- Explainer là mô hình đóng qua API, phiên bản có thể thay đổi theo thời gian nên khó tái lập tuyệt đối.
- Số câu và số vòng lặp bị giới hạn bởi thời lượng session, chưa phải quy mô đầy đủ của bài báo gốc.